# Exploración de tendencias

Abrir con `make notebook`. Usa el mismo código que la API, así que lo que pruebes aquí es lo que verá la web.

In [ ]:
import sys
sys.path.append('../backend')

import matplotlib.pyplot as plt
import pandas as pd

from app.catalog import REGION_BY_ID, VARIABLE_BY_KEY
from app.services import power, trends

## Descargar todas las variables de un punto (NASA POWER, sin clave)

In [ ]:
region = REGION_BY_ID['managua']
monthly, source = await power.get_monthly(region.lat, region.lon)
print(source, sorted(monthly))

## Tendencia de una variable

In [ ]:
r = trends.analyze(VARIABLE_BY_KEY['PRECTOTCORR'], monthly['PRECTOTCORR'], 1990, 2025)
print(r.interpretation)
for w in r.warnings:
    print('AVISO:', w)

annual = pd.DataFrame([p.model_dump() for p in r.series.annual]).set_index('year')
ax = annual['mean'].plot(marker='o', figsize=(10, 3), title=r.label)
line = pd.DataFrame([p.model_dump() for p in r.series.trend_line]).set_index('year')
line['mean'].plot(ax=ax, style='--')
plt.show()

## Escanear el punto: ¿qué está cambiando?

In [ ]:
pd.DataFrame([
    {'variable': x.label, 'cambio/década': x.slope_per_decade, 'unidad': x.unit, 'p': x.mk_p,
     'p ajustado': x.p_adjusted, 'salto': x.step_year}
    for x in trends.scan(monthly, 1990, 2025)
])